# Tutorial 05 — Créer un nouveau modèle non linéaire pairwise : Lotka-Volterra

Ce tutorial montre pas à pas comment ajouter un **nouveau modèle non linéaire pairwise** dans `awesomepkf`,
en prenant l'exemple du modèle proie-prédateur de **Lotka-Volterra** discrétisé à l'ordre 1.

**Ce que vous allez apprendre :**

1. Comprendre la structure d'un modèle pairwise scalaire (`BaseModelGxGy`, dim_x=1, dim_y=1)
2. Générer les fichiers Python du modèle pairwise et de sa version augmentée
3. Vérifier que les modèles sont automatiquement découverts par `ModelFactoryNonLinear`
4. Explorer le modèle (paramètres, équations LaTeX, jacobiens symboliques auto-générés)
5. Simuler une trajectoire et visualiser le portrait de phase
6. Appliquer les filtres EPKF, UPKF, PPF et PF

**Prérequis :** Tutorial 02 — Nonlinear Models

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

from prg import (
    NonLinear_EPKF,
    NonLinear_UPKF,
    NonLinear_PPF,
    NonLinear_PF,
    ParamNonLinear,
    ModelFactoryNonLinear,
    __version__,
)
# Voie moderne pour declarer un modele "simple" : une entree NonLinearSpec
# dans le registre NONLINEAR_CONFIGS (aucun fichier a ecrire).
from prg.models.nonlinear.configs import NONLINEAR_CONFIGS, NonLinearSpec

print(f"awesomepkf version: {__version__}")

SEED = 42
N    = 200

---
## 1. Le modèle de Lotka-Volterra pairwise

### Dynamique continue

Le modèle de Lotka-Volterra décrit l'évolution de deux populations couplées :

$$\dot{x} = \alpha\, x - \beta\, x\, y, \qquad \dot{y} = \delta\, x\, y - \gamma\, y$$

| Variable | Rôle |
|----------|------|
| $x$ | Population de proies (scalaire) |
| $y$ | Population de prédateurs (scalaire) |

| Paramètre | Rôle | Valeur (jeu C1) |
|-----------|------|--------|
| $\alpha$ | Taux de croissance des proies | 0.27503 |
| $\beta$ | Taux de prédation | 0.01030 |
| $\gamma$ | Taux de mortalité des prédateurs | 0.35974 |
| $\delta$ | Efficacité de conversion | 0.76738 |

**Point d'équilibre non trivial :** $(x^*, y^*) = (\gamma/\delta,\; \alpha/\beta) \approx (0.47,\; 26.7)$

---
### Discrétisation — intégrateur symplectique de Suris

Le schéma d'Euler explicite est **inconditionnellement instable** pour Lotka-Volterra :
les valeurs propres du Jacobien en équilibre valent $1 \pm i\sqrt{\alpha\gamma}\,\Delta t$,
de module $\sqrt{1 + \alpha\gamma\,\Delta t^2} > 1$ quelle que soit la valeur de $\Delta t$.

On utilise à la place l'**intégrateur symplectique de Suris** (volume-préservant),
qui met à jour $y$ en premier puis utilise ce $y$ pour mettre à jour $x$ :

$$y^{k+1}_{\text{det}} = y^k \cdot \exp\!\bigl((\delta\,x^k - \gamma)\,\Delta t\bigr)$$
$$x^{k+1} = x^k \cdot \exp\!\bigl((\alpha - \beta\,y^{k+1}_{\text{det}})\,\Delta t + v^x\bigr)$$
$$y^{k+1} = y^{k+1}_{\text{det}} \cdot \exp(v^y)$$

Le déterminant du Jacobien de la partie déterministe vaut exactement 1 :
les trajectoires restent sur les courbes fermées du système continu.

Le bruit est **log-normal** (il entre dans l'exponentielle), ce qui garantit $x > 0$ et $y > 0$
à chaque pas — cohérent avec les variances $\sigma^2_u, \sigma^2_v$ estimées en espace log.

---
### Formulation pairwise (`BaseModelGxGy`, dim_x=1, dim_y=1)

Dans un modèle pairwise, il y a **deux variables scalaires** ($x$ et $y$)
et **deux équations scalaires** ($g_x$ et $g_y$) :

$$g_x(x,\, y,\, v^x) = x \cdot \exp\!\bigl((\alpha - \beta\cdot y_{\text{det}})\,\Delta t + v^x\bigr), \quad y_{\text{det}} = y\cdot\exp\!\bigl((\delta x - \gamma)\Delta t\bigr)$$
$$g_y(x,\, y,\, v^y) = y_{\text{det}} \cdot \exp(v^y)$$

Le modèle LV est **naturellement pairwise** : la dynamique des proies dépend des prédateurs ($y$)
et vice-versa.

> **Note :** les jacobiens $A_n = \partial g / \partial z$ et $B_n = \partial g / \partial v$
> sont **calculés automatiquement par SymPy** dans `BaseModelGxGy`.
> Il suffit de définir `symbolic_model()`.

---
## 2. Deux façons d'ajouter un modèle

`awesomepkf` propose **deux mécanismes** pour déclarer un modèle non linéaire :

1. **Registre `NONLINEAR_CONFIGS`** (recommandé pour un modèle « simple ») —
   il suffit d'ajouter **une entrée** `NonLinearSpec` : dimensions, forme
   (`"gxgy"` pairwise ou `"fxhx"` classique), une fonction `symbolic_model`
   (SymPy en déduit les jacobiens automatiquement), et éventuellement un
   `init_hook` pour des bruits/initialisations non aléatoires. **Aucun fichier
   à écrire.**
2. **Fichier-classe** (`prg/models/nonlinear/model_*.py`) — réservé aux modèles
   qui ont besoin d'un constructeur paramétré, d'une surcharge de méthode, ou
   d'une logique augmentée par substitution. Découverts par scan du package.

Ci-dessous on utilise la **voie 1, au runtime** : on enregistre le modèle de
Lotka-Volterra pairwise directement en mémoire (cwd-indépendant, rien n'est
écrit sur le disque). Pour un modèle permanent, on copierait simplement cette
même `NonLinearSpec` dans [`prg/models/nonlinear/configs.py`](../prg/models/nonlinear/configs.py).

In [ ]:
# --- Constantes du modele proie-predateur (integrateur symplectique de Suris) ---
LV = dict(ALPHA=0.00312, BETA=0.00014, GAMMA=0.02534, DELTA=0.01175,
          SIGMAX=0.39668, SIGMAY=0.43850, DT=1.0)

def lv_symbolic_model(sx, sy, st, su):
    # gx, gy symboliques ; SymPy en deduit An = dg/dz et Bn = dg/dnoise.
    A, B, G, D, DT = LV["ALPHA"], LV["BETA"], LV["GAMMA"], LV["DELTA"], LV["DT"]
    x, y, t, u = sx[0], sy[0], st[0], su[0]
    y_det = y * sp.exp((D * x - G) * DT)                    # predateurs (sans bruit)
    sgx = sp.Matrix([x * sp.exp((A - B * y_det) * DT + t)]) # proies (bruit log-normal => x>0)
    sgy = sp.Matrix([y_det * sp.exp(u)])
    return sgx, sgy

def lv_init_hook(model):
    # Init non-aleatoire : moyenne au point d'equilibre, bruit diagonal.
    model.mQ, model.mz0, model.Pz0 = model._init_random_params(
        model.dim_x, model.dim_y, 0.10, seed=None
    )
    model.mz0 = np.array([[LV["GAMMA"] / LV["DELTA"]], [LV["ALPHA"] / LV["BETA"]]])
    model.mQ  = np.diag([LV["SIGMAX"], LV["SIGMAY"]])

DEMO_NAME = "model_x1_y1_LotkaVolterra_demo_pairwise"

# Une seule entree suffit pour ajouter le modele. attrs=... expose les
# constantes (ALPHA, ...) comme attributs de l'instance, pratique ensuite.
NONLINEAR_CONFIGS[DEMO_NAME] = NonLinearSpec(
    dim_x=1, dim_y=1, form="gxgy",
    symbolic_model=lv_symbolic_model,
    init_hook=lv_init_hook,
    attrs=dict(ALPHA=LV["ALPHA"], BETA=LV["BETA"], GAMMA=LV["GAMMA"],
               DELTA=LV["DELTA"], DT=LV["DT"]),
)
print(f"Modele '{DEMO_NAME}' enregistre au runtime (en memoire, aucun fichier ecrit).")

---
## 3. Vérifier l'enregistrement dans la factory

`ModelFactoryNonLinear.list_models()` réunit les modèles du registre
(`NONLINEAR_CONFIGS`) et les modèles fichier-classe. Le nôtre doit y figurer
et être constructible via `create`.

In [ ]:
models = ModelFactoryNonLinear.list_models()
print(f"{len(models)} modeles non lineaires disponibles ; '{DEMO_NAME}' present : {DEMO_NAME in models}")

model_lv = ModelFactoryNonLinear.create(DEMO_NAME)
print("Cree :", model_lv)

---
## 4. Explorer le modèle

### 4.1 Paramètres et informations de base

In [ ]:
model_lv = ModelFactoryNonLinear.create(DEMO_NAME)
params   = model_lv.get_params()

print(f"Modele   : {model_lv}")
print(f"dim_x    : {params['dim_x']}  (x = proies, scalaire)")
print(f"dim_y    : {params['dim_y']}  (y = predateurs, scalaire)")
print(f"Pairwise : {params['pairwiseModel']}")
print()
print(f"Point d equilibre : x* = {model_lv.GAMMA/model_lv.DELTA:.1f} (proies),  "
      f"y* = {model_lv.ALPHA/model_lv.BETA:.1f} (predateurs)")
print()
print("mz0 (etat initial moyen) [x, y] :", params['mz0'].flatten())
print()
print("mQ (covariance du bruit joint, shape", params['mQ'].shape, "):")
print(np.round(params['mQ'], 4))

### 4.2 Équations et jacobiens — générés automatiquement par SymPy

In [ ]:
from IPython.display import display, Math

display(Math(model_lv.latex_model()))

In [ ]:
import sympy as sp

print("=== gx (transition des proies) ===")
sp.pprint(model_lv._sgx, use_unicode=True)
print()
print("=== gy (dynamique des predateurs) ===")
sp.pprint(model_lv._sgy, use_unicode=True)
print()
print("=== An = dg/dz (jacobien calcule par SymPy) ===")
sp.pprint(model_lv._sAn, use_unicode=True)
print()
print("=== Bn = dg/dnoise ===")
sp.pprint(model_lv._sBn, use_unicode=True)

---
## 5. Portrait de phase — comportement déterministe

On simule le système **sans bruit** pour visualiser les orbites fermées
caractéristiques du modèle de Lotka-Volterra autour du point d'équilibre $(8, 5)$.

In [ ]:
def simulate_lv_det(x0, y0, alpha, beta, gamma, delta, dt, n_steps):
    """Simule le systeme LV discret sans bruit (integrateur symplectique de Suris)."""
    x, y = x0, y0
    traj = [(x, y)]
    for _ in range(n_steps):
        y_new = y * np.exp((delta * x - gamma) * dt)
        x_new = x * np.exp((alpha - beta * y_new) * dt)
        x, y = x_new, y_new
        traj.append((x, y))
    return np.array(traj)

alpha = model_lv.ALPHA
beta  = model_lv.BETA
gamma = model_lv.GAMMA
delta = model_lv.DELTA
dt    = model_lv.DT
x_eq, y_eq = gamma / delta, alpha / beta

# Orbites de rayon croissant autour de l equilibre (perturbations relatives)
init_conds = [(x_eq * 1.5, y_eq), (x_eq * 2.5, y_eq), (x_eq * 4.0, y_eq)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for (x0, y0), c in zip(init_conds, ["C0", "C1", "C2"]):
    traj = simulate_lv_det(x0, y0, alpha, beta, gamma, delta, dt, 600)
    ax.plot(traj[:, 0], traj[:, 1], color=c, lw=1.0)
    ax.plot(x0, y0, "o", color=c, ms=5)
ax.plot(x_eq, y_eq, "k*", ms=12, label=f"Equilibre ({x_eq:.2f}, {y_eq:.1f})")
ax.set_xlabel(r"$x$ (proies)"); ax.set_ylabel(r"$y$ (predateurs)")
ax.set_title("Portrait de phase"); ax.legend(); ax.grid(True, ls="--", alpha=0.5)

ax = axes[1]
traj = simulate_lv_det(x_eq * 2.5, y_eq, alpha, beta, gamma, delta, dt, 300)
t_ax = np.arange(len(traj)) * dt
ax.plot(t_ax, traj[:, 0], color="C0", lw=1.5, label=r"$x$ proies")
ax.plot(t_ax, traj[:, 1], color="C1", lw=1.5, label=r"$y$ predateurs")
ax.axhline(x_eq, color="C0", ls=":", alpha=0.5); ax.axhline(y_eq, color="C1", ls=":", alpha=0.5)
ax.set_xlabel("Temps"); ax.set_ylabel("Population")
ax.set_title("Serie temporelle — deterministe"); ax.legend(); ax.grid(True, ls="--", alpha=0.5)

plt.tight_layout(); plt.show()

---
## 6. Trajectoire stochastique

On simule maintenant le modèle **avec bruit** et on visualise l'état joint $(x, y)$
ainsi que le portrait de phase stochastique.

In [ ]:
def make_param(model):
    p = model.get_params().copy()
    dx = p.pop("dim_x")
    dy = p.pop("dim_y")
    return ParamNonLinear(0, dx, dy, **p)

def extract(results):
    """Extrait (x_true, x_update) depuis les resultats d un filtre.
    r[1] = vrai x, shape (dim_x, 1) = (1,1) pour LV pairwise (proies seulement).
    r[4] = x estime (Xkp1_update), shape (dim_x, 1).
    """
    xu = np.array([r[4].flatten() for r in results if r[4] is not None])
    M  = len(xu)
    xt = np.array([r[1].flatten() for r in results[:M]])
    return xt, xu

def mse(xt, xu):
    return float(np.mean((xt - xu) ** 2))

param_lv  = make_param(model_lv)
epkf_sim  = NonLinear_EPKF(param_lv, sKey=SEED)
sim_data  = epkf_sim.simulate_N_data(N)

# simulate_N_data : r[1] = vrai x (proies, dim_x=1), r[2] = vrai y (predateurs, dim_y=1)
x_true = np.array([r[1].flatten() for r in sim_data])   # (N+1, 1)
y_true = np.array([r[2].flatten() for r in sim_data])   # (N+1, 1)
z_true = np.hstack([x_true, y_true])                    # (N+1, 2)
t_sim  = np.arange(len(z_true))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(t_sim, z_true[:, 0], color="C0", lw=1.2, label=r"$x$ proies")
axes[0].axhline(x_eq, color="C0", ls=":", alpha=0.5, label=f"$x^*={x_eq}$")
axes[0].set_xlabel("Pas $k$"); axes[0].set_title(r"Proies $x^k$")
axes[0].legend(fontsize=8); axes[0].grid(True, ls="--", alpha=0.5)

axes[1].plot(t_sim, z_true[:, 1], color="C1", lw=1.2, label=r"$y$ predateurs")
axes[1].axhline(y_eq, color="C1", ls=":", alpha=0.5, label=f"$y^*={y_eq}$")
axes[1].set_xlabel("Pas $k$"); axes[1].set_title(r"Predateurs $y^k$")
axes[1].legend(fontsize=8); axes[1].grid(True, ls="--", alpha=0.5)

axes[2].plot(z_true[:, 0], z_true[:, 1], color="C3", lw=0.8, alpha=0.8)
axes[2].plot(z_true[0, 0], z_true[0, 1], "ko", ms=6, label="Depart")
axes[2].plot(x_eq, y_eq, "k*", ms=12, label=f"Eq. ({x_eq:.0f},{y_eq:.0f})")
axes[2].set_xlabel(r"$x$ proies"); axes[2].set_ylabel(r"$y$ predateurs")
axes[2].set_title("Portrait de phase stochastique")
axes[2].legend(fontsize=8); axes[2].grid(True, ls="--", alpha=0.5)

plt.tight_layout(); plt.show()

---
## 7. Filtrage — EPKF, UPKF, PPF

On applique trois filtres compatibles avec les modèles **pairwise** sur la même trajectoire.

> **Note :** `NonLinear_PF` et `NonLinear_UKF` ne supportent pas les modèles pairwise
> (`param.f` est `None`). Utiliser `NonLinear_PPF` à la place.

**Problème d'estimation :** le filtre observe $y_k$ (prédateurs, bruité)
et estime $x_k$ (proies, état caché).

- `r[1]` = vrai $x$ (proies), shape (1,1)
- `r[2]` = observation $y$ (prédateurs), shape (1,1)
- `r[4]` = $\hat{x}$ estimé, shape (1,1)

In [ ]:
epkf = NonLinear_EPKF(param_lv, sKey=SEED)
res_epkf = epkf.process_N_data(N=None, data_generator=iter(sim_data))
xt_epkf, xu_epkf = extract(res_epkf)
print(f"EPKF                   MSE = {mse(xt_epkf, xu_epkf):.6f}")

upkf = NonLinear_UPKF(param_lv, sigmaSet="wan2000", sKey=SEED)
res_upkf = upkf.process_N_data(N=None, data_generator=iter(sim_data))
xt_upkf, xu_upkf = extract(res_upkf)
print(f"UPKF (wan2000)         MSE = {mse(xt_upkf, xu_upkf):.6f}")

ppf = NonLinear_PPF(param_lv, n_particles=500, sKey=SEED)
res_ppf = ppf.process_N_data(N=None, data_generator=iter(sim_data))
xt_ppf, xu_ppf = extract(res_ppf)
print(f"PPF  (500 particules)  MSE = {mse(xt_ppf, xu_ppf):.6f}")

### 7.1 Estimées vs état vrai

In [ ]:
M   = min(len(xu_epkf), len(xu_upkf), len(xu_ppf))
t   = np.arange(M)
WIN = slice(0, min(M, 100))

# y observe (predateurs) — identique pour tous les filtres (meme sim_data)
y_obs = np.array([r[2].flatten()[0] for r in res_epkf[:M]])

filters = {
    "EPKF":        (xu_epkf[:M], "C0", "-"),
    "UPKF (wan)":  (xu_upkf[:M], "C1", "--"),
    "PPF":         (xu_ppf[:M],  "C2", "-."),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Subplot gauche : estimation de x (proies)
ax = axes[0]
ax.plot(t[WIN], xt_epkf[WIN, 0], color="black", lw=1.5, label=r"Vrai $x$")
for name, (xu_, c, ls) in filters.items():
    ax.plot(t[WIN], xu_[WIN, 0], color=c, lw=1.0, ls=ls, label=name)
ax.set_xlabel("Pas $k$"); ax.set_ylabel(r"$x$ (proies)")
ax.set_title(r"Estimation de $x$ (proies, etat cache)")
ax.legend(fontsize=8, ncol=2); ax.grid(True, ls="--", alpha=0.5)

# Subplot droit : y observe (predateurs)
ax = axes[1]
ax.plot(t[WIN], y_obs[WIN], color="C1", lw=1.0, label=r"$y_k$ observe")
ax.axhline(y_eq, color="C1", ls=":", alpha=0.5, label=f"$y^*={y_eq}$")
ax.set_xlabel("Pas $k$"); ax.set_ylabel(r"$y$ (predateurs)")
ax.set_title(r"Observation $y$ (predateurs, entree du filtre)")
ax.legend(fontsize=8); ax.grid(True, ls="--", alpha=0.5)

fig.suptitle("Comparaison des filtres — Lotka-Volterra pairwise", y=1.01)
plt.tight_layout(); plt.show()

### 7.2 MSE par filtre et par composante

In [ ]:
filter_list = [
    ("EPKF",       xt_epkf[:M], xu_epkf[:M], "C0"),
    ("UPKF (wan)", xt_upkf[:M], xu_upkf[:M], "C1"),
    ("PPF",        xt_ppf[:M],  xu_ppf[:M],  "C2"),
]

names  = [n       for n, _, _, _  in filter_list]
vals   = [float(np.mean((xt_[:, 0] - xu_[:, 0])**2)) for _, xt_, xu_, _ in filter_list]
colors = [c       for _, _, _, c  in filter_list]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(np.arange(len(names)), vals, 0.5, color=colors)
ax.set_xticks(np.arange(len(names))); ax.set_xticklabels(names)
ax.set_ylabel("MSE"); ax.set_title(r"MSE de l'estimation de $x$ (proies)")
ax.grid(True, axis="y", ls="--", alpha=0.5)
plt.tight_layout(); plt.show()

---
## 8. Récapitulatif — ajouter un modèle pairwise

| Étape | Action |
|-------|--------|
| 1 | Écrire `symbolic_model(sx, sy, st, su) -> (sgx, sgy)` (pairwise `gxgy`) — SymPy dérive les jacobiens |
| 2 | *(optionnel)* un `init_hook(model)` pour des `mQ`/`mz0`/`Pz0` non aléatoires |
| 3 | Ajouter une entrée `NonLinearSpec(...)` dans `NONLINEAR_CONFIGS` (au runtime ici, ou dans `configs.py` pour la rendre permanente) |
| 4 | `ModelFactoryNonLinear.create("<nom>")` — et c'est filtrable par EPKF / UPKF / PPF |

Pour un modèle nécessitant un constructeur paramétré ou une surcharge (ex. la
variante **augmentée**), on passe par un fichier-classe dans
`prg/models/nonlinear/` (voir `model_x1_y1_lotkavolterra_augmented.py`), que la
factory découvre par scan du package.